# Session 09 — Pandas Cleaning, Combining and Reporting
## Python Fundamentals & Practical Applications — 48-Hour Edition

**STUDENT EDITION — WRITE YOUR OWN CODE**  
**Duration:** 4 classroom hours (including a 30-minute break)  
**Audience:** complete beginners and learners transferring from spreadsheets or other programming languages

> **Beginner promise:** We inspect imperfect synthetic data, make one responsible change at a time, combine related tables, and produce a report that can be checked.
> **Professional habit:** Never hide a data-quality decision. Record what was missing, what changed, and why.

In [ ]:
student_name = input("Enter your name: ")
print(f"{student_name}, welcome to the Pandas data-quality session.")

## Learning outcomes

By the end of this session, you should be able to:

1. Inspect and convert data types.
2. Measure missing and duplicate values.
3. Clean text categories using a documented rule.
4. Handle missing values deliberately.
5. Convert dates and create date summaries.
6. Combine tables with `concat()` and `merge()`.
7. Export a checked report.

**Total: 240 minutes.** Date analysis and exploratory analysis may be challenge work.

## Google Colab quick start — ابدأ من هنا

Open the notebook in [Google Colab](https://colab.research.google.com/) or upload it using **File → Upload notebook**. Run the setup cell before the examples.

The course datasets are synthetic and are supplied in the course `datasets` folder.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    COURSE_ROOT = Path("/content/python_48h_course")
else:
    HERE = Path.cwd()
    COURSE_ROOT = HERE

DATA_DIR = COURSE_ROOT / "datasets"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



print("Dataset folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

* Visit [downgit page](https://downgit.github.io/#/home) , paste the target dataset folder URL: `https://github.com/Soliman2020/python_48h_notebooks/tree/master/datasets`, then click on `Download` button.
* On your PC, after downloading the zip file, Unzip it and upload the inner files to `dataset folder` created its directory before on the previous code cell. 

In [ ]:
REQUIRED_DATA_FILES = [
    "messy_work_items.csv",
    "energy_readings.csv",
    "sector_lookup.csv",
]
missing_files = [name for name in REQUIRED_DATA_FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing: {', '.join(missing_files)}\n"
        "Teaching datasets are missing from:\n"
        f"{DATA_DIR}.\n"
        "Add the course datasets folder/files, then run this cell again. ")
    
print("Required datasets found:", len(REQUIRED_DATA_FILES))

## 1. Inspect the mess before changing it

Cleaning is not guessing. First measure the problem. Work on a copy so the raw data remains available for comparison.

The dataset includes missing values, repeated case IDs, inconsistent text, and numeric values that need validation.

In [ ]:
raw = pd.read_csv(DATA_DIR / "messy_work_items.csv")

print("Rows and columns:", raw.shape)

In [ ]:
# Display the first few rows of the raw DataFrame to get an overview of the data
raw.head()

In [ ]:
# Display information about the DataFrame, including data types and non-null counts
raw.info()

In [ ]:
print("Missing values:")
print(raw.isna().sum())

In [ ]:
# Another way to check for missing values using isnull()
raw.isnull().sum()

In [ ]:
# Check for duplicate rows in the DataFrame
raw.duplicated().sum()

In [ ]:
# Check for duplicate case IDs in the 'case_id' column
print("Duplicate case IDs:", raw["case_id"].duplicated().sum())

In [ ]:
print("Status values, including missing values:")
print(raw["status"].value_counts(dropna=False))

In [ ]:
# Status values, as percentages:
print(raw["status"].value_counts(normalize=True).round(4) * 100)

### Guided activity 1 — make a data-quality checklist

Inspect `raw.dtypes`. Identify one column that should be numeric and one that should be a date. Which columns have missing values? Write the answers in a comment before running your code.

In [ ]:
print(raw.dtypes)
# TODO: inspect one more quality measure of your choice


## 2. Clean types, text, duplicates, and missing values

A cleaning rule must match the meaning of the field:

- Strip spaces and standardise case for categories.
- Convert numeric text with `errors="coerce"` when invalid values should become missing.
- Keep the first record for a duplicate ID only when that is the documented rule.
- Do not replace missing measurements with zero unless zero really means “measured zero”.

In [ ]:
clean = raw.drop_duplicates(subset="case_id", keep="first").copy()

clean["sector"] = clean["sector"].str.strip().str.title()

clean["status"] = clean["status"].str.strip().str.lower()

clean["hours_open"] = pd.to_numeric(clean["hours_open"], errors="coerce")

print("Rows before:", len(raw))
print("Rows after duplicate rule:", len(clean))

print("Missing hours after conversion:", clean["hours_open"].isna().sum())

In [ ]:
# Drop rows with missing hours_open values
clean = clean.dropna(subset=["hours_open"])

In [ ]:
# Check for missing values in the cleaned DataFrame
clean.isnull().sum()

### Guided activity 2 — choose a responsible missing-value action

For `hours_open`, decide whether to keep missing values, drop rows, or fill them. Explain the decision in a comment. Apply a targeted operation only if you can justify it.

In [ ]:
# TODO: document and apply your chosen missing-value rule


## 3. Dates turn records into timelines

Text that looks like a date is still text until it is parsed. Use `pd.to_datetime()` and then access components such as `.dt.year` and `.dt.month`.

A month summary can reveal changes that are difficult to see row by row.

In [ ]:
clean["created_date"] = pd.to_datetime(clean["created_date"], errors="coerce")

clean["created_month"] = clean["created_date"].dt.to_period("M")

monthly = clean.groupby("created_month").agg(cases=("case_id", "count")).reset_index()

print(monthly)

In [ ]:
# Display the last 5 rows of the cleaned DataFrame to verify the cleaning steps
clean.tail(5)

### Guided activity 3 — investigate a date field

Convert the date in `energy_readings.csv`. Print the earliest date, latest date, and number of readings per month. If the file has a different date-column name, inspect `columns` first.

In [ ]:
energy = pd.read_csv(DATA_DIR / "energy_readings.csv")
print(energy.columns.to_list())
# TODO: parse the date column and create a monthly count


## 4. Combine tables with `merge()`

A lookup table adds information to records using a shared key. A left merge keeps every row from the main table. `validate="many_to_one"` checks that the lookup key is not unexpectedly duplicated.

Before merging, standardise the key in both tables.

In [ ]:
lookup = pd.read_csv(DATA_DIR / "sector_lookup.csv")
lookup["sector"] = lookup["sector"].str.strip().str.title()
enriched = clean.merge(lookup, on="sector", how="left", validate="many_to_one")
print(enriched[["case_id", "sector", "sla_hours"]].head())
print("Rows without a matched SLA:", enriched["sla_hours"].isna().sum())
enriched["breaches_sla"] = enriched["hours_open"] > enriched["sla_hours"]

### `concat()` versus `merge()`

Use `concat()` to stack tables with the same kind of columns. Use `merge()` to add columns by matching keys. Confusing these operations can silently produce the wrong report.

### Guided activity 4 — check the join

Compare the row count before and after the merge. Investigate any rows with missing `sla_hours`. What does an unmatched key mean for the report?

In [ ]:
print("Rows before merge:", len(clean))
print("Rows after merge:", len(enriched))
print(enriched.loc[enriched["sla_hours"].isna(), ["case_id", "sector"]])

## Practice tasks


### Applied task A — `S09-E01`
Find missing values and duplicate IDs in `messy_work_items.csv`. Report the counts before changing the data.


In [ ]:
# Your code here

### Applied task B — `S09-E02`
Clean the status column and create a table of status counts.


In [ ]:
# Your code here

### Applied task C — `S09-E03`
Merge `work_items.csv` with `sector_lookup.csv`, then verify whether every work item received an SLA.


In [ ]:
# Your code here

### Challenge — `S09-E04`
Create a monthly report with total cases and average hours open. Export it as CSV.

In [ ]:
# Your code here

## Guided mini-project — Data-quality and SLA report

Build a report from `messy_work_items.csv` and `sector_lookup.csv`. Your report must:

1. Show row counts before and after the duplicate rule.
2. Show missing-value counts before cleaning.
3. Standardise the join key and perform a validated left merge.
4. Flag SLA breaches.
5. Export a summary by sector to `session_09_sla_report.csv`.
6. Explain one assumption and one limitation.

Keep the raw table unchanged and make your decisions visible in code comments.

In [ ]:
# TODO: complete the report workflow and export the summary


## Optional EDA extension

Use `describe()`, `value_counts()`, or one simple chart to investigate a question about the cleaned data. A chart should support a question; it is not a replacement for checking the table.

In [ ]:
# Optional: investigate one question about the cleaned or enriched data.


## Exit check — answer from memory

1. Why inspect missing values before cleaning?
2. What is the difference between `concat()` and `merge()`?
3. Why might `fillna(0)` be a bad decision?
4. What does `pd.to_datetime()` do?
5. What does a left merge preserve?
6. Why is `validate="many_to_one"` useful?

## Common mistakes and next steps

- Do not clean before measuring the original problem.
- Keep a raw copy when making transformations.
- Normalise join keys before merging.
- Check unmatched keys after a merge.
- Treat missing measurements as a meaning question, not merely a coding problem.
- A correct report includes assumptions and limitations.

**Next step:** use the Pandas workflow as a foundation for charts, automation, APIs, and the capstone project.